In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BigData") \
    .getOrCreate()

print("Spark started")

Spark started


Classififcation using logistic regression


In [4]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Sample Data
data = spark.createDataFrame([
    (0, 18, 2000),
    (1, 25, 3000),
    (0, 40, 5000),
    (1, 35, 7000),
    (0, 50, 8000)
], ["label", "age", "salary"])

# Feature Engineering
assembler = VectorAssembler(inputCols=["age", "salary"], outputCol="features")
df = assembler.transform(data)

# Split Data
train, test = df.randomSplit([0.8, 0.2])

# Model
lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train)

# Predictions
predictions = model.transform(test)
predictions.select("features", "label", "prediction").show()

# Evaluation
evaluator = BinaryClassificationEvaluator(labelCol="label")
accuracy = evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

+-------------+-----+----------+
|     features|label|prediction|
+-------------+-----+----------+
|[40.0,5000.0]|    0|       0.0|
+-------------+-----+----------+

Accuracy: 0.0


Clustering using k-means


In [5]:
from pyspark.ml.clustering import KMeans

# Sample Data
data = spark.createDataFrame([
    (1, 1.0, 2.0),
    (2, 1.5, 1.8),
    (3, 5.0, 8.0),
    (4, 8.0, 8.0),
    (5, 1.0, 0.6),
    (6, 9.0, 11.0)
], ["id", "x", "y"])

# Feature Vector
assembler = VectorAssembler(inputCols=["x", "y"], outputCol="features")
df = assembler.transform(data)

# KMeans Model
kmeans = KMeans(k=2, seed=1)
model = kmeans.fit(df)

# Predictions
predictions = model.transform(df)
predictions.select("id", "features", "prediction").show()

# Cluster Centers
centers = model.clusterCenters()
print("Cluster Centers:", centers)

+---+----------+----------+
| id|  features|prediction|
+---+----------+----------+
|  1| [1.0,2.0]|         1|
|  2| [1.5,1.8]|         1|
|  3| [5.0,8.0]|         0|
|  4| [8.0,8.0]|         0|
|  5| [1.0,0.6]|         1|
|  6|[9.0,11.0]|         0|
+---+----------+----------+

Cluster Centers: [array([7.33333333, 9.        ]), array([1.16666667, 1.46666667])]


Recommendation using ALS


In [6]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Sample User-Item Ratings
data = spark.createDataFrame([
    (0, 0, 4.0),
    (0, 1, 3.0),
    (1, 0, 2.0),
    (1, 1, 5.0),
    (2, 1, 4.0)
], ["userId", "itemId", "rating"])

# Split Data
train, test = data.randomSplit([0.8, 0.2])

# ALS Model
als = ALS(
    maxIter=10,
    regParam=0.1,
    userCol="userId",
    itemCol="itemId",
    ratingCol="rating",
    coldStartStrategy="drop"
)

model = als.fit(train)

# Predictions
predictions = model.transform(test)
predictions.show()

# Evaluation
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print("RMSE:", rmse)

# Recommendations
user_recs = model.recommendForAllUsers(2)
user_recs.show(truncate=False)

+------+------+------+----------+
|userId|itemId|rating|prediction|
+------+------+------+----------+
|     0|     0|   4.0|0.16492188|
|     1|     1|   5.0|0.29824668|
+------+------+------+----------+

RMSE: 4.290355955775489
+------+---------------------------------+
|userId|recommendations                  |
+------+---------------------------------+
|0     |[{1, 2.9488454}, {0, 0.16492188}]|
|1     |[{0, 1.9074912}, {1, 0.29824668}]|
|2     |[{1, 3.9317937}, {0, 0.2198958}] |
+------+---------------------------------+



In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
